# NITK-UsoundSim - full relay (phantom to beamformed image)
Run top to bottom in Colab or Jupyter. Start Jupyter inside integration/02_full_relay (the first cell adds the module paths).

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)
from usoundsim_full import run_full, overview_figure, demo_points
from usoundsim_relay import RelayConfig
from receiver_beamforming import RxConfig, locate_peak

## 1. Nine wire targets

In [ ]:
res = run_full('points', RelayConfig(), RxConfig())
for x, _, z in res.relay.scatterers.xyz:
    xb, zb = locate_peak(res.bf, x, z)
    print(f'true ({x*1e3:5.1f},{z*1e3:5.1f}) mm  error ({(xb-x)*1e3:+.3f},{(zb-z)*1e3:+.3f}) mm')
overview_figure(res, 'points'); plt.show()

## 2. A tissue preset (change the name: generic, abdomen, kidney, thyroid, breast, cardiac, carotid, msk, obstetric, brain, lung, prostate)

In [ ]:
res = run_full('generic', RelayConfig(max_scatterers=3000), RxConfig(apodization='hann', f_number=1.5))
overview_figure(res, 'generic'); plt.show()
print(res.bf.notes)

## 3. Your own points
Columns `x_mm,z_mm,amp`:

In [ ]:
pts = np.array([[0, 15, 1.0], [4, 25, 0.8], [-5, 32, 0.6]])
res = run_full(pts, RelayConfig(), RxConfig(), units='mm')
overview_figure(res, 'my points'); plt.show()

## 4. Compare apodisations on the same channel data (runner 5 only)

In [ ]:
import receiver_beamforming as rb
fig, ax = plt.subplots(1, 3, figsize=(13, 5))
for a, apod in zip(ax, ['rect', 'hann', 'blackman']):
    b = rb.beamform(res.bf.input, RxConfig(apodization=apod))
    a.imshow(b.bmode_db, extent=b.extent_mm, cmap='gray', vmin=-50, vmax=0, aspect='auto'); a.set_title(apod)
plt.show()